# Hierarchical dementia model with vascular-focused features (UDS v3)
This notebook trains:
- Stage 1: **NoDementia_5y vs Dementia-within-5y**
- Stage 2a: **Vascular vs non-vascular** among dementia
- Stage 2b: **PD/LBD vs AD** among non-vascular dementia

It is designed to improve **Vascular** performance by ensuring vascular-brain-injury and vascular-risk variables are included.


In [1]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM import failed:", e)

RANDOM_STATE = 42
OUT_DIR = "../output"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Load cohort ----
# Expect a cohort.csv produced by your cohort-building notebook.
COHORT_PATH = os.path.join(OUT_DIR, "cohort.csv")
df = pd.read_csv(COHORT_PATH)
print("Loaded:", COHORT_PATH, "shape=", df.shape)

# Basic sanity
assert "label" in df.columns, "cohort.csv must contain a 'label' column"


Loaded: ../output\cohort.csv shape= (38341, 35)


## 1) Ensure vascular-focused variables are present
If any are missing, go back to the cohort-building notebook and add them to `starter_features` so they end up in `cohort.csv`.

**UDS v3 vascular brain injury form (D1) variables to include if available:**
- `CVDIF` (vascular brain injury: primary/contributing/non-contributing)
- `STROKDEC` (stroke temporally related to cognitive decline)
- `STKIMAG` (stroke confirmed by imaging)
- `INFNETW` (imaging infarct in cognitive networks)
- `INFWMH` (extensive WMH + exec dysfunction)

**UDS health history (A5) vascular risk variables to include if available:**
- `CBSTROKE` (stroke history)
- `CBTIA` (TIA history) *(name varies; include if present)*
- `HYPERTEN`, `DIABETES`, `HYPERCHO`
- `CVAFIB`, `CVCHF`, `CVANGINA`, `CVBYPASS`, `CVANGIO`, `CVHVALVE`, `CVOTHR`

**UDS medications (A4) that often proxy vascular risk treatment:**
- `NACCLIPL` (lipid-lowering), `NACCAC` (anticoagulant/antiplatelet), `NACCBETA`, `NACCCCBS`, `NACCDIUR`, `NACCANGI`


In [2]:

# Candidate vascular feature set (only keeps those that actually exist in your cohort.csv)
vas_vars = [
    # D1 VBI / imaging evidence
    "CVDIF","STROKDEC","STKIMAG","INFNETW","INFWMH",
    # A5 health history (vascular risk)
    "CBSTROKE","CBTIA","HYPERTEN","DIABETES","HYPERCHO",
    "CVAFIB","CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
    # A4 meds proxies
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
]

present_vas = [c for c in vas_vars if c in df.columns]
missing_vas = [c for c in vas_vars if c not in df.columns]

print("Present vascular-focused vars (count={}):".format(len(present_vas)))
print(present_vas)

print("\nMissing vascular-focused vars (count={}):".format(len(missing_vas)))
print(missing_vas)


Present vascular-focused vars (count=6):
['CBSTROKE', 'CBTIA', 'HYPERTEN', 'DIABETES', 'HYPERCHO', 'CVAFIB']

Missing vascular-focused vars (count=18):
['CVDIF', 'STROKDEC', 'STKIMAG', 'INFNETW', 'INFWMH', 'CVCHF', 'CVANGINA', 'CVBYPASS', 'CVANGIO', 'CVHVALVE', 'CVOTHR', 'NACCLIPL', 'NACCAC', 'NACCBETA', 'NACCCCBS', 'NACCDIUR', 'NACCANGI', 'NACCVASD']


## 2) Train/test split and preprocessing
We train on the 4 outcome classes used in your earlier experiments:
`NoDementia_5y`, `AD`, `PD_LBD`, `Vascular`.


In [3]:

KEEP = {"AD", "NoDementia_5y", "PD_LBD", "Vascular"}
df4 = df[df["label"].isin(KEEP)].copy()
df4["label"] = df4["label"].astype(str)

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Split labels and features
y = df4["label"].astype(str)

drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
X_all = df4.drop(columns=drop_cols)

# --- Clean common special missing codes ---
# Many UDS code vars use -4 for NA / skip patterns, and 9 for "unknown" in some forms.
SPECIAL_MISS = {-4, 9, 99}

for c in X_all.columns:
    if pd.api.types.is_numeric_dtype(X_all[c]) or X_all[c].dtype == object:
        s = pd.to_numeric(X_all[c], errors="ignore")
        # only mask numeric columns
        if pd.api.types.is_numeric_dtype(s):
            X_all[c] = s.mask(s.isin(list(SPECIAL_MISS)), np.nan)

# Treat these as categorical-coded columns (extend as needed)
CAT_CODE_VARS = [c for c in ["SEX","RACE","HISPANIC","NACCUDSD"] if c in X_all.columns]
for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    X_all[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

# Identify categorical vs numeric
cat_cols = [c for c in X_all.columns if (X_all[c].dtype == object)]
num_cols = [c for c in X_all.columns if c not in cat_cols]

print("Numeric cols:", len(num_cols), "Categorical cols:", len(cat_cols))

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


4-class rows: 14569
label
NoDementia_5y    11083
AD                3013
PD_LBD             289
Vascular           184
Name: count, dtype: int64


ValueError: invalid error value specified

## 3) Hierarchical model (Stage1 + Stage2a + Stage2b)
Tune thresholds at the bottom:
- `T_DEM` (dementia gate)
- `T_VAS` (vascular gate)
- `T_PD` (PD gate)


In [ ]:

if not HAS_LGBM:
    raise RuntimeError("Install lightgbm to run this notebook.")

# -----------------------------
# Stage 1: Binary Dementia within 5y?
# -----------------------------
y1_train = (y_train != "NoDementia_5y").astype(int)
y1_test  = (y_test  != "NoDementia_5y").astype(int)

w1_train = np.where(y1_train.values == 1, 3.0, 1.0).astype(np.float64)

stage1 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LGBMClassifier(
        objective="binary",
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE
    ))
])
stage1.fit(X_train, y1_train, model__sample_weight=w1_train)
p_dem = stage1.predict_proba(X_test)[:, 1]

print("\n=== Stage 1 (Binary): Dementia within 5y? ===")
pred1 = (p_dem >= 0.5).astype(int)
print(classification_report(y1_test, pred1, digits=4, target_names=["NoDementia", "Dementia"]))
print("Balanced accuracy:", balanced_accuracy_score(y1_test, pred1))
print(confusion_matrix(y1_test, pred1))

# -----------------------------
# Stage 2a: Vascular vs Non-vascular among dementia
# -----------------------------
dem_train_mask = (y_train != "NoDementia_5y")
X2_train = X_train.loc[dem_train_mask].copy()
y2_train = y_train.loc[dem_train_mask].copy()

y_vas_train = (y2_train == "Vascular").astype(int)
w_vas = np.where(y_vas_train.values == 1, 40.0, 1.0).astype(np.float64)

stage2a_vas = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE
    ))
])
stage2a_vas.fit(X2_train, y_vas_train, model__sample_weight=w_vas)

# -----------------------------
# Stage 2b: PD_LBD vs AD among NON-vascular dementia
# -----------------------------
nonvas_train_mask = dem_train_mask & (y_train != "Vascular")
X3_train = X_train.loc[nonvas_train_mask].copy()
y3_train = y_train.loc[nonvas_train_mask].copy()

bad = sorted(set(y3_train.unique()) - set(["AD", "PD_LBD"]))
if bad:
    raise ValueError(f"Unexpected labels in non-vascular dementia subset: {bad}")

y_pd_train = (y3_train == "PD_LBD").astype(int)
w_pd = np.where(y_pd_train.values == 1, 10.0, 1.0).astype(np.float64)

stage2b_pd = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE
    ))
])
stage2b_pd.fit(X3_train, y_pd_train, model__sample_weight=w_pd)

# -----------------------------
# Combine into 4-class prediction
# -----------------------------
label_order_4 = ["NoDementia_5y", "AD", "PD_LBD", "Vascular"]

p_vas_all = stage2a_vas.predict_proba(X_test)[:, 1]   # ~ P(Vascular | Dementia)
p_pd_all  = stage2b_pd.predict_proba(X_test)[:, 1]    # ~ P(PD | Dementia, NonVas)

# Conditional subtype probabilities given dementia:
p_ad_cond_all = (1.0 - p_vas_all) * (1.0 - p_pd_all)
p_pd_cond_all = (1.0 - p_vas_all) * p_pd_all

# Joint probabilities
probs = np.zeros((len(X_test), 4), dtype=float)
probs[:, 0] = 1.0 - p_dem
probs[:, 3] = p_dem * p_vas_all
probs[:, 2] = p_dem * p_pd_cond_all
probs[:, 1] = p_dem * p_ad_cond_all

# ---------- THRESHOLDS (start here) ----------
T_DEM = 0.45
T_VAS = 0.05
T_PD  = 0.35

pred_h = np.full(len(X_test), "NoDementia_5y", dtype=object)
is_dem = (p_dem >= T_DEM)
idx = np.where(is_dem)[0]
if len(idx) > 0:
    pred_sub = np.full(len(idx), "AD", dtype=object)

    vas_ok = (p_vas_all[idx] >= T_VAS)
    pd_ok  = (p_pd_all[idx]  >= T_PD)

    pred_sub[pd_ok] = "PD_LBD"
    pred_sub[vas_ok] = "Vascular"  # vascular wins if it passes its gate
    pred_h[idx] = pred_sub

print("\n=== Hierarchical model (vascular-focused features) 4-class ===")
print(classification_report(y_test, pred_h, digits=4, labels=label_order_4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_h))
print("Confusion matrix labels:", label_order_4)
print(confusion_matrix(y_test, pred_h, labels=label_order_4))

# Save for analysis
prob_df = pd.DataFrame(probs, columns=[f"p_{c}" for c in label_order_4], index=X_test.index)
prob_df["p_dem"] = p_dem
prob_df["p_ad_cond"] = p_ad_cond_all
prob_df["p_pd_cond"] = p_pd_cond_all
prob_df["p_vas_cond"] = p_vas_all
prob_df["y_true"] = y_test.values
prob_df["y_pred"] = pred_h
prob_path = os.path.join(OUT_DIR, "hierarchical_vasfocused_pred_probs.csv")
prob_df.to_csv(prob_path, index=True)
print("Saved:", prob_path)
